# Collections and Lambdas

Organize Kotlin data in collections and use small functions to select, transform, and search it.

We can now represent one note as an object. Most apps manage many objects: messages, reminders, products, or saved places. This lesson builds on familiar Java and C++ collection ideas while introducing the compact Kotlin operations commonly used to prepare app data.

## Learning Goals

- Choose a list, set, or map and recognize read-only versus mutable operations.
- Write lambdas that select data with `filter` and transform it with `map`.
- Handle a missing `find` result and use `forEach` for actions such as printing.

## Why This Matters

An app often stores more data than a screen should display. A reminders screen might show only unfinished items, while a search screen needs a useful response when nothing matches.

Collection operations let you express those rules directly. Separating the source data from a display result helps preserve information that another screen still needs. Without that separation, hiding a completed item could accidentally remove it from the app's source collection. These same rules will help us build and test a small app model before connecting it to Android screens.

## Check Your Starting Point

Lesson 3 introduced `data class Note(val title: String, val archived: Boolean = false)`. Suppose `original` is `Note("Travel")` and `changed` is `original.copy(archived = true)`. Write both archive values and explain whether the original object changed.

In [ ]:
Your response:
Write your explanation here.

<details>
<summary>Show answer</summary>

The original is still `false`; the copy is `true`. `copy` creates another instance with selected constructor-property values replaced. The original remains available. We will keep that distinction between source and result while processing collections.

</details>

## Video Demonstration

Watch a list of notes become a list of active titles. Then see how a search returns a missing result without crashing the program.

<video controls preload="metadata" width="800" aria-label="Collections and Lambdas demonstration">
<source src="media/04_collections_and_lambdas/lesson.mp4" type="video/mp4">
<track kind="captions" src="media/04_collections_and_lambdas/captions.vtt" srclang="en" label="English">
Your browser does not support embedded video.
</video>

[Read the video transcript and visual description](media/04_collections_and_lambdas/transcript.md).

## Concept

### Choose a Collection and Its Operations

A **collection** groups values. A **list** keeps an ordered sequence and permits duplicates. A **set** represents unique elements. A **map** associates each unique key with a value, such as a setting name with its stored text. These are the same broad data-structure ideas you know from Java and C++.

Kotlin provides both **read-only collection APIs**, which expose reading operations, and **mutable collection APIs**, which also expose changes. An API here is the set of operations available through a type. `listOf` creates a list with a read-only API; `mutableListOf` gives you a mutable list. These names are library functions, not language keywords.

```kotlin
val savedTitles = mutableListOf("Travel", "Groceries")
savedTitles.add("Study")
```

`add` appends an item; `remove` removes a matching item. The `size` property reports the number of items. A `val` prevents replacing the `savedTitles` binding, but the mutable object still supports changes.

In [ ]:
val savedTitles = mutableListOf("Travel", "Groceries")
savedTitles.add("Study")
savedTitles.remove("Groceries")
println(savedTitles)
println(savedTitles.size)

This prints `[Travel, Study]` and `2`. Removing `Groceries` changed the mutable list itself. That is different from the selection operations we will use shortly.

Read-only does not promise that an object can never change through another reference. `List<String>` says that elements are strings and the reference exposes the read-only list API. The following two names refer to the same underlying list.

In [ ]:
val readOnlyTitles: List<String> = savedTitles
savedTitles.add("Weekend")
println(readOnlyTitles)
println(readOnlyTitles.size)

The read-only reference now sees `[Travel, Study, Weekend]` and a size of `3`. It did not make a frozen snapshot. Calling `readOnlyTitles.add(...)` is unavailable through its type, but the mutable reference can still change the shared list. Also, a read-only collection can hold objects with mutable properties.

An empty list has no elements from which to infer a type. Supply the expected element type when creating it: `val emptyTitles: List<String> = listOf()`. After declaring a `Reminder` data class, `val reminders: List<Reminder> = listOf()` similarly creates an empty list of reminders. Filtering or mapping an empty list returns an empty result list; searching it with `find` returns null.

### Sets and Maps in Small App Tasks

Use a set when duplicates have no meaning. The `in` operator tests membership, while `size` counts unique elements. Use a map when you want to retrieve a value by a key. `mapOf("theme" to "dark")` uses the library function `to` to form a key-value pair; `to` is not a Kotlin keyword. Brackets perform a key lookup.

A map lookup can return null when the key is missing. The safe handling from Lesson 2 still applies. A map collection and the `map` transformation introduced below share a word, but perform different jobs.

In [ ]:
val tags = setOf("study", "home", "study")
println(tags.size)
println("study" in tags)
val settings = mapOf("theme" to "dark", "language" to "English")
println(settings["theme"] ?: "system")
println(settings["font"] ?: "standard")

The output is `2`, `true`, `dark`, and `standard`. The set counts `study` once. The map contains `theme`, but has no `font` key, so the last lookup uses its fallback. These checks do not depend on a set's iteration order.

### Describe a Small Operation with a Lambda

A **lambda expression** is a function written as a value, often for a short operation passed to another function. Curly braces enclose it. Parameters appear before `->`; the final expression in the body supplies the result.

```kotlin
val normalizeTitle = { title: String -> title.uppercase() }
```

Here `title` receives one string, and the body produces uppercase text. Kotlin infers the type of the lambda value. Calling `normalizeTitle("Travel")` invokes that operation, just as parentheses invoke a named function. Defining the lambda alone does not run its body.

In [ ]:
val normalizeTitle = { title: String -> title.uppercase() }
println(normalizeTitle("Travel"))
println(normalizeTitle("Study"))

The results are `TRAVEL` and `STUDY`. Each call supplies a new argument. The operation creates uppercase strings; it does not change a string in place.

### Select Elements with filter

The **filter operation** builds a list containing the source elements that pass a test. Its lambda is a **predicate**, a function that answers true or false for one element. For each element, true means keep it and false means omit it from the result.

For the simple text used here, a string's `length` property gives its character count. The test below keeps titles with at most four characters. Some Unicode symbols use more than one unit in this count, so it is not a universal count of visible symbols.

```kotlin
val shortTitles = labels.filter({ title -> title.length <= 4 })
```

The surrounding call tells Kotlin that `title` is a string, so its type need not be repeated. When the final argument is a lambda, Kotlin permits **trailing-lambda syntax**: move the braces outside the parentheses. With no other arguments, omit those empty parentheses:

```kotlin
val shortTitles = labels.filter { title -> title.length <= 4 }
```

For a lambda with one inferred parameter, you can omit the parameter declaration and use its implicit name, **`it`**:

```kotlin
val shortTitles = labels.filter { it.length <= 4 }
```

All three forms describe the same test. `it` names the current element inside this lambda; it is not a keyword or a special variable available everywhere. A descriptive parameter name is often clearer for a longer expression.

In [ ]:
val labels = listOf("Home", "Weekend", "Work")
val shortTitles = labels.filter { it.length <= 4 }
println(shortTitles)
println(labels)

This prints `[Home, Work]`, then `[Home, Weekend, Work]`. The result keeps the passing elements in source order. The original list still has three elements. `filter` selects references to elements; it does not make a deep copy of each object. If no elements pass, the result is an empty list.

### Transform Elements with map

The **map transformation** produces one result for each source element. Its lambda describes the new value. Unlike `filter`, it changes what each result contains rather than deciding which elements to keep.

```kotlin
val loudTitles = shortTitles.map { it.uppercase() }
```

The input is the previously selected list. This operation creates uppercase display strings and returns another list. It does not overwrite the strings or replace the source list. A transformation can also change the element type, such as turning each note object into its title string.

In [ ]:
val loudTitles = shortTitles.map { it.uppercase() }
println(loudTitles)
println(shortTitles)

The results are `[HOME, WORK]` and `[Home, Work]`. Two source elements produced two transformed results. Combining selection and transformation keeps each rule small: first choose the needed data, then choose its display form.

### Find One Match and Handle Its Absence

The **find operation** returns the first element that passes its predicate, or null if no element passes. It is a search for one element, not a list of all matches. That is why its result needs the nullable handling you already learned.

For `labels.find { it == "Home" }`, the result is the existing string `Home`. A search for `Missing` returns null. Using `?.` followed by `?:` makes both outcomes explicit.

In [ ]:
val foundLabel = labels.find { it == "Home" }
val absentLabel = labels.find { it == "Missing" }
println(foundLabel?.uppercase() ?: "No matching title")
println(absentLabel?.uppercase() ?: "No matching title")

The output is `HOME` and `No matching title`. On the second line, the safe call skips uppercase conversion and the Elvis operator selects the fallback. Avoid using `!!` merely to silence the nullable type; an ordinary no-match result should not crash an app.

### Perform an Action with forEach

The **forEach operation** runs an action for each element. Printing is a **side effect**, an action that affects something beyond producing a return value. `forEach` is useful when that action is your goal; it does not build a list of transformed values.

Use `map` to produce a list you will keep. Use `forEach` when you want to perform an action for every element. A normal `for` loop is also a clear choice for sequential actions.

In [ ]:
loudTitles.forEach { title -> println("Show: $title") }

This prints `Show: HOME` and `Show: WORK`. Each call to `println` happens once for an element. The original list is unchanged because the action only prints; an action that deliberately modified an object could have a different effect.

<details class="animation-panel" open>
<summary>Follow selection and transformation — show or hide animation</summary>
<p><img src="media/04_collections_and_lambdas/filter_map.gif" alt="Three source notes include archived Old list. Filtering retains Lab ideas and App sketch; mapping produces their title strings. The original source still contains all three notes." width="960" style="max-width:100%;height:auto;"></p>
</details>

The sequence separates the original notes, the selected note objects, and their display strings. Filtering omits an archived note from the result without deleting it from the source. The 10-second animation loops; closing its panel hides the motion.

[View the labeled still diagram](media/04_collections_and_lambdas/filter_map_still.png).

## Worked Example

### Prepare Active Note Titles

Our notes model now needs to prepare active titles for a screen and supply a useful message when a search finds nothing.

1. A `Note` data class gives every object a title and an archive flag.
2. `listOf` groups three notes, including one archived note.
3. `filter { !it.archived }` keeps notes whose archive flag is false. `!` negates the Boolean flag, so true means this note is active.
4. `map { it.title }` turns each retained note into its title string.
5. `forEach` prints those strings. The later `find` searches the original list for a particular title.
6. `match?.title ?: "No matching note"` safely supplies display text for either search outcome.

Notice the meaning of `it` changes with the collection's element type: the filter and first map receive a `Note`; the printing lambda receives a `String`.

In [ ]:
data class Note(val title: String, val archived: Boolean = false)
val notes = listOf(Note("Lab ideas"), Note("Old list", true), Note("App sketch"))
val activeNotes = notes.filter { !it.archived }
val titles = activeNotes.map { it.title }
titles.forEach { println(it) }
val match = notes.find { it.title == "Missing" }
println(match?.title ?: "No matching note")

The output is:

```text
Lab ideas
App sketch
No matching note
```

`Old list` is absent from the display titles because it is archived. It still belongs to the original three-note list. The selected list contains two note objects, and the mapped list contains two strings in the same order. The search has no matching title, so its nullable result becomes a clear message.

This is data preparation for an app. It does not create an Android screen or permanently save anything.

## Guided Practice

### Follow the selected counts

Before running the next cell, predict both printed lists. Trace which input counts pass the filter and what the map adds to each selected count. Explain why the two copies of `2` remain in the first printed list.

In [ ]:
Your prediction:
First printed list:
Second printed list:
Selection and transformation:
Why the source still contains both 2 values:


In [ ]:
val previewCounts = listOf(2, 5, 2, 8)
val largerCounts = previewCounts.filter { it > 3 }
val nextCounts = largerCounts.map { count -> count + 1 }
println(previewCounts)
println(nextCounts)

<details>
<summary>Show answer</summary>

```text
[2, 5, 2, 8]
[6, 9]
```

Only `5` and `8` pass `it > 3`. The map adds one to each, producing `6` and `9` in the same order. `it` and the named parameter `count` each refer to one element in their own lambda. The source list keeps both `2` values because `filter` and `map` return results instead of removing or replacing source entries. A list allows repeated values.

</details>

### Complete a label pipeline

Start with `practiceActions = listOf("Read", "", "Sketch")`. Complete these steps in order:

1. Use `filter` to keep nonempty strings in `visibleActions`.
2. Use `map` with a named lambda parameter, `action`, to create `"Next: $action"` labels in `actionLabels`.
3. Use `forEach` to print each label.

Expected output:

```text
Next: Read
Next: Sketch
```

Then modify only the filter rule so it keeps `"Sketch"`. Run again; the only output should be `Next: Sketch`. Keep the mapping and printing steps unchanged.

In [ ]:
val practiceActions = listOf("Read", "", "Sketch")
// TODO: Select nonempty actions, transform them into labels, and print each label.
// Then modify only the selection rule to keep Sketch.


<details>
<summary>Show answer</summary>

```kotlin
val practiceActions = listOf("Read", "", "Sketch")
val visibleActions = practiceActions.filter { it != "" }
val actionLabels = visibleActions.map { action -> "Next: $action" }
actionLabels.forEach { println(it) }
```

The filter lambda returns a Boolean: true keeps that string. The map lambda returns the new label for each retained string. Its final expression supplies the result, so no explicit `return` is needed. `forEach` prints the labels; it does not build the label list.

For the modification, replace the filter condition with `it == "Sketch"`. The remaining steps now receive only that one string, so they print `Next: Sketch`. Filtering after adding the prefix would require a different condition; keep the requested order.

</details>

### Repair the collection choice

This snippet is intentionally faulty. It should append `"Sketch"` and print `[Read, Sketch]`, but it does not compile:

```kotlin
val practiceQueue = listOf("Read")
practiceQueue.add("Sketch")
println(practiceQueue)
```

Explain why `add` is unavailable. Then write the corrected snippet in the Kotlin cell. Keep the variable declared with `val`, and keep the `add` call. Explain why changing only `val` to `var` would not fix the problem.

In [ ]:
Your diagnosis:
Why add is unavailable:
Why changing only val to var would not fix it:


In [ ]:
// TODO: Rewrite the faulty snippet with a collection that supports add.
// Keep val, then print the resulting queue.


<details>
<summary>Show answer</summary>

```kotlin
val practiceQueue = mutableListOf("Read")
practiceQueue.add("Sketch")
println(practiceQueue)
```

`listOf` exposes a read-only list API, so `add` is unavailable through that value. `mutableListOf` provides an editable list and its `add` operation. The `val` binding still refers to the same list after adding an element. Changing only to `var` would allow assigning a different list to the variable, but it would not add mutation operations to a read-only list API. Read-only access is not a general promise that no other reference can change a collection.

</details>

## Independent Practice

### Build a reminder display pipeline

Create a small program from these requirements. Use separate variables for each result so you can inspect each step.

- Define `PracticeReminder` as a data class with `title: String` and `completed: Boolean = false`, both `val` properties.
- Declare `practiceReminders: List<PracticeReminder>` containing `Read` (unfinished), `Submit` (completed), and `Sketch` (unfinished), in that order. Set `wantedTitle` to `"Sketch"`.
- Filter unfinished reminders, then map them to `"Next: "` followed by the title. Print the resulting list of labels.
- Map the original list to its titles. Print `Source titles: ` followed by that list. The completed title must still be present.
- Use `find` on the original list to search for `wantedTitle`. Print the found title, or `No matching reminder` when nothing matches. Use a safe call and Elvis fallback.

The initial output must be:

```text
[Next: Read, Next: Sketch]
Source titles: [Read, Submit, Sketch]
Sketch
```

Test the same program twice more by editing only the inputs and rerunning the cell:

1. Keep the three reminders but change `wantedTitle` to `"Missing"`. The first two lines stay the same; the last becomes `No matching reminder`.
2. Keep that missing search title and replace the source with `val practiceReminders: List<PracticeReminder> = listOf()`. This explicitly typed empty list has no reminder elements. Expect:

```text
[]
Source titles: []
No matching reminder
```

The empty output list is `[]`; do not print a made-up reminder. Keep your selection, transformation, and search logic unchanged for both tests.

In [ ]:
// TODO: Define PracticeReminder, supply the inputs, and build the display pipeline.
// Run the normal, missing-title, and empty-source cases by editing only the inputs.


<details>
<summary>Show answer</summary>

```kotlin
data class PracticeReminder(val title: String, val completed: Boolean = false)
val practiceReminders: List<PracticeReminder> = listOf(
    PracticeReminder("Read"),
    PracticeReminder("Submit", completed = true),
    PracticeReminder("Sketch")
)
val wantedTitle = "Sketch"
val unfinishedReminders = practiceReminders.filter { !it.completed }
val reminderLabels = unfinishedReminders.map { "Next: ${it.title}" }
println(reminderLabels)
val sourceTitles = practiceReminders.map { it.title }
println("Source titles: $sourceTitles")
val foundReminder = practiceReminders.find { it.title == wantedTitle }
println(foundReminder?.title ?: "No matching reminder")
```

`filter` retains whole reminder objects whose completed flag is false. `map` then produces strings for those selected objects. The original list still contains `Submit`; the source-title line makes that visible. Searching the original list is separate from selecting unfinished reminders.

For the missing-title test, change only `wantedTitle` to `"Missing"`. `find` returns null, the safe call produces null, and Elvis supplies `No matching reminder`. For the empty-source test, also replace the list declaration and its three entries with `val practiceReminders: List<PracticeReminder> = listOf()`. The explicit element type lets Kotlin type the empty list. Both mapped lists are empty and `find` returns null, so the output is:

```text
[]
Source titles: []
No matching reminder
```

Avoid `!!`: a missing match is an expected input here. Keep and use the results of `filter` and `map`; calling either and discarding its result does not update the source list.

</details>

### Explain your test evidence

After running your program, cite the output that shows filtering left the source entries intact. Then explain how the missing-title and empty-source runs test different situations. Name the operation that returns null in both cases and the expression that provides the fallback.

In [ ]:
Your explanation after testing:
Evidence the source retained its entries:
Missing-title case versus empty-source case:
Nullable operation and fallback expression:


<details>
<summary>Show answer</summary>

`Source titles: [Read, Submit, Sketch]` still contains the completed reminder even though the display list excludes it. In the missing-title run there are source entries, but none has the requested title. In the empty-source run there are no entries to inspect, so the two displayed lists are also empty. `find` returns null in both cases. `foundReminder?.title ?: "No matching reminder"` safely supplies the fallback. These tests catch solutions that assume the list always has an item or the search always succeeds.

</details>

## Summary

- Lists preserve order and allow duplicates; sets represent unique values; maps associate keys with values.
- A read-only API restricts the operations through a reference. It does not guarantee a deeply immutable object or a frozen snapshot.
- A lambda supplies behavior as a value. A final lambda can follow a call; a single inferred parameter may be named `it`.
- `filter` selects elements, while `map` transforms them. These operations return new result lists without changing the source collection themselves.
- `find` returns one matching element or null. Handle the no-match case explicitly.
- `forEach` performs actions; it is not a replacement for `map` when you need a transformed list.

Next, we will pass actions into our own functions and read several compact Kotlin patterns often found in Android code.

## Reflection

A reminders app lets a user switch between all reminders and unfinished reminders. Explain why keeping the original list and deriving a display list is useful. Describe when you would need a mutable collection API, and why declaring its variable with `val` would still be reasonable.

In [ ]:
Your reflection:
Why keep the original list:
When an editable collection is needed:
Why val can still fit:


<details>
<summary>Show answer</summary>

A derived display list can hide completed reminders without deleting them, so switching back to all reminders can show them again. Adding a new reminder to an existing collection requires a mutable API. A `val` variable can still refer to that editable collection; it prevents replacing the binding while permitting operations supplied by the collection. A read-only view limits what callers can do through that view, but does not prove the underlying collection can never change.

</details>

## Supplemental Reading

- [Kotlin collection overview](https://kotlinlang.org/docs/collections-overview.html) — lists, sets, maps, and read-only versus mutable interfaces.
- [Kotlin lambdas](https://kotlinlang.org/docs/lambdas.html) — lambda parameters, implicit `it`, and trailing-lambda syntax.
- [Kotlin collection filtering](https://kotlinlang.org/docs/collection-filtering.html) — selecting elements by a predicate.
- [Kotlin collection transformations](https://kotlinlang.org/docs/collection-transformations.html) — creating transformed results with `map`.
- [Kotlin element retrieval](https://kotlinlang.org/docs/collection-elements.html) — finding an element and handling missing results.